In [1]:
import xgboost as xgb

from pathlib import Path
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd

import os

import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
from pyearthtools.data.time import Petdt
from pyearthtools.pipeline.operations.xarray.join import GeospatialTimeSeriesMerge
import site_archive_nci

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import shap

In [2]:
# data = pd.read_parquet('/scratch/er8/cd3022/xgb_datasets/syd_gridpoint_2024-01.parquet')

data = pd.read_parquet('/scratch/er8/cd3022/xgb_datasets/syd_radiances_2024-01-01.parquet')

In [13]:
X = data.drop(columns=['surface_global_irradiance'])
for n in range(1, 11):
    X = X.drop(columns=[f'surface_global_irradiance_t{n}'])

y = data['surface_global_irradiance_t3']

In [14]:
split_index = int(len(data) * 0.8)

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

In [15]:
scaler = StandardScaler()
scaler.set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
model = xgb.XGBRegressor(
    random_state=42,
    # device='cuda'
)
model.fit(X_train_scaled, y_train,
        eval_set=[(X_train_scaled, y_train), (X_test_scaled, y_test)],
       verbose=False)

# Predict on test set
y_pred = model.predict(X_test_scaled)

# Quick evaluation of model performance using correlation and RMSE
correlation = np.corrcoef(y_test, y_pred)[0, 1]
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Correlation between predicted and actual values: {correlation:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")

Correlation between predicted and actual values: 0.566
Root Mean Squared Error (RMSE): 143.623


In [17]:
model.save_model("/scratch/er8/cd3022/xgb_models/radiance_model.json")